# Análisis de backbones — Queratocono (Orbscan) · Sano (0) vs Queratocono (1)

**Proyecto Final Integrador** · Etapa 02, posterior al EDA (`01_EDA_queratocono.ipynb`).

**Objetivo de decisión.** Clasificar cada **ojo** como *sano* (0) o *queratocono* (1) a partir de los
4 mapas Orbscan (Anterior, Axial, Pachymetry, Posterior), con una prioridad explícita: **no dejar
pasar un queratocono** (falso negativo). Un QC no detectado puede terminar en cirugía refractiva
y ectasia, mientras que un falso positivo cuesta un estudio de más.

## Qué se prueba
| Fase | Contenido |
|---|---|
| 1-2 | Manifest por **ojo** (1,407), corrección `3E+132`→`3E132`, asserts anti-fuga, inspección OD/OS |
| 4 | **Baseline tabular** (regresión logística con variables clínicas): referencia de qué tan difícil es el problema |
| 5 | **Análisis de backbones**: embeddings congelados de MobileNetV2 / EfficientNet-B3 / ConvNeXt-Tiny × 4 mapas → LogReg / SVM, más fusión de mapas |
| 6 | Transfer learning (solo la cabeza) del mejor backbone, por mapa, **con y sin augmentación** |
| 7 | **Fine-tuning** de los últimos bloques, con comparación antes/después |
| 8 | Fusión de mapas + umbral `tau` calibrado en **out-of-fold** + tabla final |
| 9-10 | **Grad-CAM** antes/después del FT y métricas por género/ojo |

## Métricas (por qué no alcanza solo el recall ni solo el AUC)
* **Selección** (épocas, checkpoints, backbone): **ROC-AUC de validación**. Es independiente del umbral y estable. El recall, con ~100 QC por fold, salta ~1 % por ojo y sobreajusta la selección.
* **Punto de operación**: `tau` = mayor umbral con **sensibilidad ≥ 0.95 sobre las predicciones out-of-fold** de los 5 folds (~417 QC). Se predice QC si `P(QC) ≥ tau`. Así se maximiza la especificidad.
* **Reporte primario en test**: **sensibilidad @tau (IC Wilson 95 %) + especificidad @tau**. La sensibilidad sola se maximiza trivialmente con "todo QC"; la especificidad muestra el costo.
* **Si el AUC satura (~0.99)**: se desempata con la **especificidad a sensibilidad 0.95** (región de alta sensibilidad de la ROC) y un **IC bootstrap agrupado por paciente** (dos ojos del mismo paciente no son independientes).
* Secundarias: PR-AUC (AP), balanced accuracy, F2 y Brier (calibración).

## Ajustes respecto del pipeline de Mandioca
* **Splits oficiales** del dataset (`split_80_20` + `fold` 1-5, agrupados por paciente): CV 5-fold sobre train, test intacto.
* **Augmentación acotada**: `RandomResizedCrop(0.85-1)` + rotación ±5° + traslación leve. **Sin** VFlip (la asimetría inferior/superior es un signo de QC), **sin** rotación grande y **sin** ColorJitter (el color codifica el valor del mapa).
* **Ojos OS espejados** horizontalmente (`ESPEJAR_OS`): Orbscan muestra OD como T–N y OS como N–T. Así todos quedan en la misma orientación anatómica.
* Problema binario: un logit + `BCEWithLogits(pos_weight = n_sanos / n_QC del fold)`.

## Cómo ejecutar
* **Colab (corrida completa)**: *Entorno de ejecución → GPU* y *Ejecutar todo*. Los datos se descargan solos de Zenodo (CornOrb, registro abierto 20542091, ~680 MB, con verificación md5) a `/content` y se descomprimen allí. Las salidas quedan en `MyDrive/PFI_queratocono/salidas/02_backbone/` y se reutilizan como caché si se corta la sesión (`USAR_DRIVE = False` para no montar Drive).
* **Local**: si no encuentra los datos en `DATA_DIR`, `Datos/zenodo` o `../../..`, los descarga a `Datos/zenodo` (ignorada por git).
* **Smoke test** (corrida mínima para validar que nada falle): variable de entorno `SMOKE=1`, o `SMOKE = True` en la celda siguiente. Usa 2 folds, 6 ojos por clase, 1 época y solo MobileNetV2. Los artefactos llevan el sufijo `_smoke`.

## Fase 0 — Entorno, semillas y configuración

In [ ]:
import os, sys, json, time, math, random, zipfile, subprocess, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    try:
        import timm  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
from torchvision import transforms as T
from PIL import Image

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve

warnings.filterwarnings("ignore", category=UserWarning, module="timm")

SMOKE = os.environ.get("SMOKE", "0") == "1"    # para forzar en Colab: SMOKE = True
SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
NUM_WORKERS = 2 if EN_COLAB else 0
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

print("entorno:", "Colab" if EN_COLAB else "local", "| torch", torch.__version__, "| timm", timm.__version__)
print("device:", DEVICE, "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("modo SMOKE:", SMOKE)

In [ ]:
OBJ_SENS = 0.95                      # sensibilidad exigida (out-of-fold) al calibrar tau
MAPAS = ["Anterior", "Axial", "Pachymetry", "Posterior"]
BACKBONES = {                        # nombre timm -> tamaño de entrada
    "mobilenetv2_100": 224,
    "efficientnet_b3.ra2_in1k": 300,
    "convnext_tiny.fb_in1k": 224,
}
CORTO = {"mobilenetv2_100": "mnv2", "efficientnet_b3.ra2_in1k": "effb3", "convnext_tiny.fb_in1k": "cnxt"}
FOLDS = [1, 2, 3, 4, 5]
ESPEJAR_OS = True                    # OS (N-T) -> misma orientación que OD (T-N)
AUG_BRAZOS = [False, True]           # transfer learning (cabeza) sin / con augmentación
HEAD_CFG = dict(epochs=20, patience=5, batch=32, lr=1e-3)
FT_CFG = dict(epochs=15, patience=5, batch=16, aug=True, unfreeze_blocks=3,
              lr_backbone=1e-4, lr_head=3e-4, weight_decay=1e-2)
MAPAS_FT = list(MAPAS)               # mapas a los que se aplica fine-tuning (recortar si falta tiempo de GPU)
C_LR_TAB, C_LR_EMB = 1.0, 0.1        # regularización L2 (fija, no se tunea con test)
N_BOOT = 1000                        # bootstrap agrupado por paciente
GRADCAM_POR_CLASE = 2
N_SMOKE = 6                          # ojos por clase y por fold en modo SMOKE

if SMOKE:
    BACKBONES = {"mobilenetv2_100": 224}
    FOLDS = [1, 2]
    HEAD_CFG.update(epochs=1, patience=1, batch=8)
    FT_CFG.update(epochs=1, patience=1, batch=4)
    N_BOOT, GRADCAM_POR_CLASE = 50, 1

TAG = "_smoke" if SMOKE else ""
print(json.dumps(dict(BACKBONES=BACKBONES, FOLDS=FOLDS, HEAD_CFG=HEAD_CFG, FT_CFG=FT_CFG,
                      OBJ_SENS=OBJ_SENS, N_BOOT=N_BOOT, TAG=TAG), indent=1))

In [ ]:
import hashlib
import urllib.request

# CornOrb en Zenodo. La versión 17127265 es de acceso restringido; esta (20542091) es abierta,
# CC BY 4.0, con los mismos archivos. Los md5 son los que publica Zenodo.
ZENODO_RECORD = "20542091"
ZENODO_ARCHIVOS = {
    "clinical_data_and_labels.csv": "983fda79c62fbe54d7fa9e83f440cfee",
    "ORBSCAN_Dataset.zip": "4afca93d00abf1c5e1fe25ea681a4e17",   # ~680 MB
}
# En Colab, Drive solo guarda las salidas (caché de resultados y checkpoints) para que
# sobrevivan si se corta la sesión. Con False todo queda en /content y se pierde al cerrar.
USAR_DRIVE = True


def md5(path, bloque=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(bloque), b""):
            h.update(b)
    return h.hexdigest()


def descargar_zenodo(destino):
    """Descarga el CSV y el zip de CornOrb a `destino` y verifica su md5. Si ya están, no hace nada."""
    destino.mkdir(parents=True, exist_ok=True)
    for nombre, esperado in ZENODO_ARCHIVOS.items():
        f = destino / nombre
        if f.exists() and md5(f) == esperado:
            continue
        url = f"https://zenodo.org/api/records/{ZENODO_RECORD}/files/{nombre}/content"
        print(f"descargando {nombre} de Zenodo ...")
        tmp = f.with_suffix(f.suffix + ".part")
        with urllib.request.urlopen(url) as r, open(tmp, "wb") as out:
            total = int(r.headers.get("Content-Length", 0))
            leido, ultimo = 0, -1
            while chunk := r.read(1 << 22):
                out.write(chunk)
                leido += len(chunk)
                pct = int(100 * leido / total) if total else 0
                if pct // 10 != ultimo:
                    ultimo = pct // 10
                    print(f"  {pct:3d} % ({leido / 1e6:.0f} MB)")
        assert md5(tmp) == esperado, f"md5 incorrecto en {nombre}: borrar {tmp} y reintentar"
        tmp.replace(f)
    return destino


def es_dir_datos(p):
    return p is not None and (p / "clinical_data_and_labels.csv").exists() and \
        ((p / "ORBSCAN_Dataset").is_dir() or (p / "ORBSCAN_Dataset.zip").exists())


if EN_COLAB and USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

# Primero se busca una copia ya descargada; si no hay, se baja de Zenodo
CANDIDATOS = [os.environ.get("DATA_DIR"), "Datos/zenodo", "../Datos/zenodo", "../../..",
              "/content/datos_zenodo", "/content/drive/MyDrive/PFI_queratocono"]
DATA_DIR = next((Path(c).resolve() for c in CANDIDATOS if c and es_dir_datos(Path(c))), None)
if DATA_DIR is None:
    DATA_DIR = descargar_zenodo(Path("/content/datos_zenodo") if EN_COLAB else Path("Datos/zenodo").resolve())
CSV_PATH = DATA_DIR / "clinical_data_and_labels.csv"

IMG_ROOT = DATA_DIR / "ORBSCAN_Dataset"
if EN_COLAB and (DATA_DIR / "ORBSCAN_Dataset.zip").exists():
    IMG_ROOT = Path("/content/ORBSCAN_Dataset")       # disco local de Colab: mucho más rápido que Drive
    if not IMG_ROOT.is_dir():
        with zipfile.ZipFile(DATA_DIR / "ORBSCAN_Dataset.zip") as z:
            z.extractall("/content")
elif not IMG_ROOT.is_dir():
    with zipfile.ZipFile(DATA_DIR / "ORBSCAN_Dataset.zip") as z:
        z.extractall(DATA_DIR)
assert IMG_ROOT.is_dir(), IMG_ROOT

if os.environ.get("OUT_DIR"):
    OUT = Path(os.environ["OUT_DIR"])
elif EN_COLAB:
    base = Path("/content/drive/MyDrive/PFI_queratocono") if USAR_DRIVE else Path("/content")
    OUT = base / "salidas" / "02_backbone"
else:
    OUT = Path("salidas") / "02_backbone"
for sub in ("res", "ckpt", "emb", "fig"):
    (OUT / sub).mkdir(parents=True, exist_ok=True)
print("datos  :", DATA_DIR)
print("mapas  :", IMG_ROOT)
print("salidas:", OUT.resolve())

## Fase 1 — Datos: manifest por ojo, limpieza y verificación de fugas

* **Unidad de muestra = ojo.** El CSV tiene 1,454 filas, pero solo hay 1,407 carpetas de ojo con 4 PNG. Las filas repetidas son mediciones del mismo ojo que **comparten las mismas imágenes**. Se agrupa por `(patient_code, eye)` y las variables clínicas se promedian (solo se usan en el baseline tabular).
* `3E+132` es la versión de Excel de `3E132` (la carpeta). Se corrige antes del join.
* Variables clínicas: los Kmax < 30 D (4.0 / 4.9 / 5.7, físicamente imposibles) pasan a NaN y se imputan con la mediana del train de cada fold. Los ojos con paquimetría mínima > central se marcan y se conservan. Los ejes (0-180°) se codifican como `sin(2θ)` y `cos(2θ)`.

In [ ]:
df_raw = pd.read_csv(CSV_PATH, dtype={"patient_code": str})
corr = df_raw["patient_code"].str.contains("E+", regex=False)
print("códigos con notación científica corregidos:", sorted(df_raw.loc[corr, "patient_code"].unique()))
df_raw["patient_code"] = df_raw["patient_code"].str.replace("E+", "E", regex=False)

df_raw["kmax_invalido"] = df_raw["kmax_value_D"] < 30
df_raw.loc[df_raw["kmax_invalido"], "kmax_value_D"] = np.nan
df_raw["flag_pachy"] = df_raw["pachy_thinnest_um"] > df_raw["pachy_central_um"]
for col, nom in [("astig_axis_deg", "astig"), ("kmax_axis_deg", "kmax")]:
    rad = np.deg2rad(2 * df_raw[col])
    df_raw[f"{nom}_sin"], df_raw[f"{nom}_cos"] = np.sin(rad), np.cos(rad)
print(f"filas CSV: {len(df_raw)} | Kmax inválidos -> NaN: {int(df_raw.kmax_invalido.sum())} "
      f"| paquimetría mínima > central: {int(df_raw.flag_pachy.sum())}")

VARS_TAB = ["age_years", "astig_value_D", "kmax_value_D", "pachy_central_um", "pachy_thinnest_um",
            "pachy_thinnest_x", "pachy_thinnest_y", "asphericity_anterior", "asphericity_posterior",
            "astig_sin", "astig_cos", "kmax_sin", "kmax_cos"]

g = df_raw.groupby(["patient_code", "eye"], sort=True)
consist = g[["label", "split_80_20", "fold", "gender"]].nunique(dropna=False)
assert (consist <= 1).all().all(), "label/split/fold/gender no son constantes dentro de un ojo"

man = g.agg(**{c: (c, "mean") for c in VARS_TAB},
            label=("label", "first"), gender=("gender", "first"),
            split=("split_80_20", "first"), fold=("fold", "first"),
            n_mediciones=("label", "size"), kmax_invalido=("kmax_invalido", "any"),
            flag_pachy=("flag_pachy", "any")).reset_index()
man["fold"] = man["fold"].fillna(0).astype(int)          # 0 = test
man["label"] = man["label"].astype(int)
man["eye_key"] = man["patient_code"] + "_" + man["eye"]
for m in MAPAS:
    man[f"rel_{m}"] = man.patient_code + "/" + man.eye + "/" + man.patient_code + "_" + man.eye + "_" + m + ".png"

# --- verificaciones ---
carpetas = {(p.parent.name, p.name) for p in IMG_ROOT.glob("*/*") if p.is_dir()}
assert carpetas == set(zip(man.patient_code, man.eye)), "CSV y carpetas no coinciden"
faltan = [r for m in MAPAS for r in man[f"rel_{m}"] if not (IMG_ROOT / r).exists()]
assert not faltan, f"faltan {len(faltan)} PNG, p.ej. {faltan[:3]}"
assert len(man) == 1407, len(man)
pac = man.groupby("patient_code").agg(n_split=("split", "nunique"), n_fold=("fold", "nunique"),
                                      n_lab=("label", "nunique"))
assert (pac.n_split == 1).all(), "paciente en train y test a la vez (fuga)"
assert (pac.n_fold == 1).all(), "paciente en más de un fold (fuga entre folds)"
assert (pac.n_lab == 1).all()
assert set(man.loc[man.split == "train", "fold"]) == {1, 2, 3, 4, 5}
assert (man.loc[man.split == "test", "fold"] == 0).all()

print(f"OK: {len(man)} ojos / {man.patient_code.nunique()} pacientes | "
      f"{int((man.n_mediciones > 1).sum())} ojos con mediciones repetidas (promediadas)")
display(pd.crosstab(man["split"].where(man.split == "test", "fold " + man.fold.astype(str)),
                    man["label"].map({0: "sano", 1: "QC"}), margins=True))
man.drop(columns=[f"rel_{m}" for m in MAPAS]).to_csv(OUT / f"manifest_ojos{TAG}.csv", index=False)

In [ ]:
MAN_FULL = man.copy()
if SMOKE:
    partes = [man[(man.split == "train") & (man.fold == k)].groupby("label").sample(n=N_SMOKE, random_state=SEED)
              for k in FOLDS]
    partes.append(man[man.split == "test"].groupby("label").sample(n=N_SMOKE, random_state=SEED))
    man = pd.concat(partes)
else:
    man = man[(man.split == "test") | man.fold.isin(FOLDS)]
man = man.sort_values(["patient_code", "eye"]).reset_index(drop=True)

Y = man["label"].values.astype(int)
IDX_POOL = np.where(man.split == "train")[0]          # train (CV) -> predicciones out-of-fold
FOLD_POOL = man["fold"].values[IDX_POOL]
IDX_TEST = np.where(man.split == "test")[0]
Y_POOL, Y_TEST = Y[IDX_POOL], Y[IDX_TEST]
PAC_TEST = man["patient_code"].to_numpy(dtype=str)[IDX_TEST]
print(f"ojos en uso: {len(man)} | train (CV, {len(FOLDS)} folds): {len(IDX_POOL)} "
      f"(QC={int(Y_POOL.sum())}) | test: {len(IDX_TEST)} (QC={int(Y_TEST.sum())})")

## Fase 2 — Inspección visual de los 4 mapas (OD vs OS)

Se muestran un ojo sano y uno con QC, OD y OS, tal como se los entrega al modelo (OS espejado si `ESPEJAR_OS`).
Hay que tener en cuenta que el mapa **Axial trae valores dióptricos impresos**: Grad-CAM (Fase 9) permite ver si el modelo mira el mapa o "lee" los números.

In [ ]:
def cargar_png(rel, eye):
    im = Image.open(IMG_ROOT / rel).convert("RGB")
    if ESPEJAR_OS and eye == "OS":
        im = im.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
    return im


ejemplos = []
for lab in (0, 1):
    ambos = MAN_FULL[MAN_FULL.label == lab].groupby("patient_code").filter(lambda d: set(d.eye) == {"OD", "OS"})
    pc = ambos.patient_code.iloc[0]
    ejemplos += [ambos[(ambos.patient_code == pc) & (ambos.eye == e)].iloc[0] for e in ("OD", "OS")]

fig, axes = plt.subplots(len(ejemplos), 4, figsize=(14, 3.6 * len(ejemplos)))
for i, r in enumerate(ejemplos):
    for j, m in enumerate(MAPAS):
        rel = f"{r.patient_code}/{r.eye}/{r.patient_code}_{r.eye}_{m}.png"
        axes[i, j].imshow(cargar_png(rel, r.eye))
        axes[i, j].set_title(f"{'QC' if r.label else 'sano'} | {r.eye} | {m}", fontsize=9)
        axes[i, j].axis("off")
fig.suptitle(f"Mapas Orbscan (ESPEJAR_OS={ESPEJAR_OS})", fontsize=13)
plt.tight_layout()
plt.savefig(OUT / "fig" / f"inspeccion_mapas{TAG}.png", dpi=90)
plt.show()

## Fase 3 — Infraestructura: métricas, umbral y registro de resultados

Todo experimento produce **probabilidades out-of-fold (OOF)** para el train y la **probabilidad
promedio de los modelos de cada fold** para el test. Con eso se calibra `tau` (solo OOF), se
evalúa el test sin tocarlo y se pueden fusionar mapas o modelos promediando probabilidades.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    den = 1 + z ** 2 / n
    centro = (p + z ** 2 / (2 * n)) / den
    half = z * math.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / den
    return centro - half, centro + half


def elegir_tau(y, p, objetivo=OBJ_SENS):
    """Mayor tau (-> mayor especificidad) con sensibilidad >= objetivo. Regla: QC si p >= tau."""
    pos = np.sort(np.asarray(p, float)[np.asarray(y) == 1])[::-1]
    if len(pos) == 0:
        return 0.5
    return float(pos[int(math.ceil(objetivo * len(pos))) - 1])


def esp_a_sens(y, p, objetivo=OBJ_SENS):
    """Especificidad máxima alcanzable con sensibilidad >= objetivo (sin umbral fijado)."""
    if len(np.unique(y)) < 2:
        return float("nan")
    fpr, tpr, _ = roc_curve(y, p, drop_intermediate=False)
    ok = tpr >= objetivo - 1e-12
    return float(1 - fpr[ok].min()) if ok.any() else float("nan")


def metricas_bin(y, p, tau):
    y = np.asarray(y).astype(int)
    p = np.asarray(p, float)
    pred = (p >= tau).astype(int)
    tp = int(((y == 1) & (pred == 1)).sum()); fn = int(((y == 1) & (pred == 0)).sum())
    tn = int(((y == 0) & (pred == 0)).sum()); fp = int(((y == 0) & (pred == 1)).sum())
    sens = tp / (tp + fn) if tp + fn else float("nan")
    esp = tn / (tn + fp) if tn + fp else float("nan")
    prec = tp / (tp + fp) if tp + fp else 0.0
    f2 = 5 * prec * sens / (4 * prec + sens) if (4 * prec + sens) > 0 else 0.0
    dos = len(np.unique(y)) == 2
    return dict(tau=float(tau), sens=sens, sens_ic=list(wilson(tp, tp + fn)), esp=esp,
                esp_ic=list(wilson(tn, tn + fp)), bal_acc=(sens + esp) / 2, f2=float(f2),
                tp=tp, fn=fn, tn=tn, fp=fp,
                auc=float(roc_auc_score(y, p)) if dos else float("nan"),
                ap=float(average_precision_score(y, p)) if dos else float("nan"),
                brier=float(brier_score_loss(y, p)), esp_a_sens=esp_a_sens(y, p))


def boot_ci(y, p, pacientes, tau, B=None, seed=SEED):
    """IC 95 % por bootstrap re-muestreando PACIENTES (los 2 ojos de un paciente van juntos)."""
    B = B or N_BOOT
    rng = np.random.default_rng(seed)
    y, p = np.asarray(y), np.asarray(p, float)
    ids = np.unique(pacientes)
    grupos = {i: np.where(pacientes == i)[0] for i in ids}
    acc = {"auc": [], "sens": [], "esp": []}
    for _ in range(B):
        sel = np.concatenate([grupos[i] for i in rng.choice(ids, len(ids), replace=True)])
        ys, ps = y[sel], p[sel]
        if len(np.unique(ys)) < 2:
            continue
        pred = ps >= tau
        acc["auc"].append(roc_auc_score(ys, ps))
        acc["sens"].append(pred[ys == 1].mean())
        acc["esp"].append((~pred[ys == 0]).mean())
    return {k: ([float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] if v else [float("nan")] * 2)
            for k, v in acc.items()}

In [ ]:
RESULTADOS = {}


def ruta_res(exp_id):
    return OUT / "res" / f"{exp_id}{TAG}.json"


def cargar_res(exp_id):
    f = ruta_res(exp_id)
    if f.exists():
        r = json.loads(f.read_text(encoding="utf-8"))
        if r.get("eye_keys_pool") == man["eye_key"].to_numpy(dtype=str)[IDX_POOL].tolist():
            print(f"[cache] {exp_id}")
            return r
    return None


def armar_res(exp_id, familia, oof, p_test, extra=None):
    oof, p_test = np.asarray(oof, float), np.asarray(p_test, float)
    assert not np.isnan(oof).any() and len(oof) == len(IDX_POOL)
    auc_folds = [float(roc_auc_score(Y_POOL[FOLD_POOL == k], oof[FOLD_POOL == k])) for k in FOLDS]
    tau = elegir_tau(Y_POOL, oof)
    res = dict(id=exp_id, familia=familia, extra=extra or {},
               auc_folds=auc_folds, auc_cv_media=float(np.mean(auc_folds)), auc_cv_sd=float(np.std(auc_folds)),
               oof=metricas_bin(Y_POOL, oof, tau), test=metricas_bin(Y_TEST, p_test, tau),
               test_boot=boot_ci(Y_TEST, p_test, PAC_TEST, tau),
               oof_p=np.round(oof, 6).tolist(), test_p=np.round(p_test, 6).tolist(),
               eye_keys_pool=man["eye_key"].to_numpy(dtype=str)[IDX_POOL].tolist())
    ruta_res(exp_id).write_text(json.dumps(res, indent=1), encoding="utf-8")
    return res


def fusionar(exp_id, ids, familia, extra=None):
    """Fusión tardía: promedio de probabilidades (OOF y test) de experimentos ya corridos."""
    rs = [RESULTADOS[i] for i in ids]
    return armar_res(exp_id, familia, np.mean([r["oof_p"] for r in rs], 0),
                     np.mean([r["test_p"] for r in rs], 0), dict(extra or {}, componentes=ids))


def fila(r):
    o, t, b, e = r["oof"], r["test"], r["test_boot"], r["extra"]
    return dict(ID=r["id"], familia=r["familia"], backbone=e.get("backbone", "-"), mapa=e.get("mapa", "-"),
                AUC_cv=r["auc_cv_media"], AUC_cv_sd=r["auc_cv_sd"], AUC_oof=o["auc"], tau=o["tau"],
                Sens_oof=o["sens"], Esp_oof=o["esp"],
                AUC_test=t["auc"], AUC_ic=f"[{b['auc'][0]:.3f}-{b['auc'][1]:.3f}]",
                Sens_test=t["sens"], Sens_ic_wilson=f"[{t['sens_ic'][0]:.3f}-{t['sens_ic'][1]:.3f}]",
                Sens_ic_boot=f"[{b['sens'][0]:.3f}-{b['sens'][1]:.3f}]",
                Esp_test=t["esp"], Esp_ic_boot=f"[{b['esp'][0]:.3f}-{b['esp'][1]:.3f}]",
                Esp_a_sens95_test=t["esp_a_sens"], AP_test=t["ap"], F2_test=t["f2"], Brier_test=t["brier"],
                FN_test=t["fn"], FP_test=t["fp"])


def tabla(ids):
    return pd.DataFrame([fila(RESULTADOS[i]) for i in ids if i in RESULTADOS])


def cv_sklearn(exp_id, familia, X, make_clf, extra=None):
    r = cargar_res(exp_id)
    if r:
        return r
    oof, p_test = np.full(len(IDX_POOL), np.nan), []
    for k in FOLDS:
        tr, va = IDX_POOL[FOLD_POOL != k], FOLD_POOL == k
        clf = make_clf().fit(X[tr], Y[tr])
        oof[va] = clf.predict_proba(X[IDX_POOL[va]])[:, 1]
        p_test.append(clf.predict_proba(X[IDX_TEST])[:, 1])
    return armar_res(exp_id, familia, oof, np.mean(p_test, 0), extra)

## Fase 4 — Baseline tabular (variables clínicas)

Regresión logística L2 estandarizada (`class_weight="balanced"`) con las 13 variables clínicas.
**No** se usa `gender` (EDA §7: no hay asociación a nivel de paciente y se evita un atajo
demográfico). Sirve como **referencia**: si las imágenes no superan a Kmax + paquimetría, el
aporte del backbone hay que justificarlo por otro lado (automatización, explicabilidad).

In [ ]:
X_TAB = man[VARS_TAB].values.astype(float)


def make_lr_tab():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(C=C_LR_TAB, class_weight="balanced", max_iter=5000))


RESULTADOS["TAB_LR"] = cv_sklearn("TAB_LR", "baseline tabular", X_TAB, make_lr_tab,
                                  dict(backbone="-", mapa="clínicas", clf="LogReg"))
display(tabla(["TAB_LR"]).round(4))

coef = make_lr_tab().fit(X_TAB[IDX_POOL], Y_POOL)[-1].coef_[0]
display(pd.Series(coef, index=VARS_TAB, name="coef. estandarizado (ajuste sobre todo el train)")
        .sort_values(key=np.abs, ascending=False).round(3).to_frame())

## Fase 5 — Análisis de backbones: embeddings congelados × mapa

Para cada backbone preentrenado en ImageNet y cada mapa:

1. Se extrae el vector del **GAP del backbone congelado** (sin augmentación). Se cachea en `.npz` y se verifica su alineación con el manifest.
2. Se evalúa con **LogReg L2** y con **SVM RBF calibrada**, en CV de 5 folds con los folds oficiales.
3. Se prueban dos **fusiones de mapas**: concatenar los 4 embeddings (LogReg), y promediar las probabilidades de las 4 LogReg por mapa.

El backbone ganador (por **AUC OOF de la fusión**, con desempate por especificidad OOF) pasa a las Fases 6-7.

In [ ]:
_CACHE_IMG = {}


def imagenes_mapa(mapa, size):
    """Lista (alineada con man) de imágenes pre-redimensionadas a size*1.07 (OS espejado). Caché en RAM por mapa."""
    pre = int(round(size * 1.07))
    if any(k[0] != mapa for k in _CACHE_IMG):
        _CACHE_IMG.clear()
    if (mapa, pre) not in _CACHE_IMG:
        _CACHE_IMG[(mapa, pre)] = [cargar_png(rel, eye).resize((pre, pre), Image.BILINEAR)
                                   for rel, eye in zip(man[f"rel_{mapa}"], man["eye"])]
    return _CACHE_IMG[(mapa, pre)]


def build_tf(size, aug, mean, std):
    if aug:
        ops = [T.RandomResizedCrop(size, scale=(0.85, 1.0), ratio=(0.95, 1.05)),
               T.RandomAffine(degrees=5, translate=(0.03, 0.03), fill=255)]
    else:
        ops = [T.CenterCrop(size)]
    return T.Compose(ops + [T.ToTensor(), T.Normalize(mean, std)])


class MapaDS(Dataset):
    def __init__(self, imgs, idx, tf):
        self.imgs, self.idx, self.tf = imgs, np.asarray(idx), tf

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = self.idx[i]
        return self.tf(self.imgs[j]), np.float32(Y[j])


class Clf(nn.Module):
    def __init__(self, bb_name):
        super().__init__()
        self.backbone = timm.create_model(bb_name, pretrained=True, num_classes=0)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(self.backbone.num_features, 1))
        cfg = self.backbone.pretrained_cfg
        self.mean, self.std = tuple(cfg["mean"]), tuple(cfg["std"])

    def train(self, mode=True):
        super().train(mode)
        if mode:
            self.backbone.eval()      # BN/dropout del backbone siempre en eval (batches chicos)
        return self

    def forward(self, x):
        return self.head(self.backbone(x)).squeeze(1)


@torch.no_grad()
def predecir(model, ds, bs=64):
    model.eval()
    out = []
    for x, _ in DataLoader(ds, batch_size=bs, shuffle=False, num_workers=NUM_WORKERS):
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            out.append(torch.sigmoid(model(x.to(DEVICE)).float()).cpu().numpy())
    return np.concatenate(out)


@torch.no_grad()
def extraer_emb(bb, size, mapa):
    f = OUT / "emb" / f"emb_{CORTO[bb]}_{mapa}{TAG}.npz"
    if f.exists():
        d = np.load(f, allow_pickle=False)
        if d["eye_key"].tolist() == man["eye_key"].to_numpy(dtype=str).tolist():
            return d["X"]
    t0 = time.time()
    model = Clf(bb).to(DEVICE).eval()
    ds = MapaDS(imagenes_mapa(mapa, size), np.arange(len(man)), build_tf(size, False, model.mean, model.std))
    feats = []
    for x, _ in DataLoader(ds, batch_size=32, shuffle=False, num_workers=NUM_WORKERS):
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            feats.append(model.backbone(x.to(DEVICE)).float().cpu().numpy())
    X = np.concatenate(feats).astype(np.float32)
    assert np.isfinite(X).all()
    np.savez_compressed(f, X=X, eye_key=man["eye_key"].to_numpy(dtype=str))
    print(f"  emb {CORTO[bb]} {mapa}: {X.shape} ({time.time() - t0:.0f}s)")
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return X


EMB = {}
for mapa in MAPAS:                      # mapa afuera: reutiliza la caché de imágenes
    for bb, size in BACKBONES.items():
        EMB[(bb, mapa)] = extraer_emb(bb, size, mapa)
print({f"{CORTO[b]}|{m}": v.shape for (b, m), v in EMB.items()})

In [ ]:
def make_lr_emb():
    return make_pipeline(StandardScaler(), LogisticRegression(C=C_LR_EMB, class_weight="balanced", max_iter=5000))


def make_svm_emb():
    return make_pipeline(StandardScaler(), CalibratedClassifierCV(
        SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced", random_state=SEED),
        cv=3, method="sigmoid"))


IDS_EMB = []
for bb in BACKBONES:
    c = CORTO[bb]
    for mapa in MAPAS:
        for nom, mk in [("LR", make_lr_emb), ("SVM", make_svm_emb)]:
            eid = f"EMB_{nom}_{c}_{mapa}"
            RESULTADOS[eid] = cv_sklearn(eid, f"embeddings + {nom}", EMB[(bb, mapa)], mk,
                                         dict(backbone=bb, mapa=mapa, clf=nom))
            IDS_EMB.append(eid)
    eid = f"EMB_LR_{c}_concat4"
    RESULTADOS[eid] = cv_sklearn(eid, "embeddings concat 4 mapas + LR",
                                 np.concatenate([EMB[(bb, m)] for m in MAPAS], 1), make_lr_emb,
                                 dict(backbone=bb, mapa="concat4", clf="LR"))
    IDS_EMB.append(eid)
    eid = f"EMB_LR_{c}_fusprom"
    RESULTADOS[eid] = fusionar(eid, [f"EMB_LR_{c}_{m}" for m in MAPAS], "embeddings LR, promedio 4 mapas",
                               dict(backbone=bb, mapa="prom4", clf="LR"))
    IDS_EMB.append(eid)

tab_emb = tabla(IDS_EMB)
display(tab_emb[["ID", "backbone", "mapa", "AUC_cv", "AUC_cv_sd", "AUC_oof", "Esp_oof",
                 "AUC_test", "Sens_test", "Esp_test", "Esp_a_sens95_test"]].round(4))

In [ ]:
piv = tab_emb[tab_emb.ID.str.startswith("EMB_LR")].pivot_table(index="backbone", columns="mapa", values="AUC_oof")
fig, ax = plt.subplots(figsize=(1.6 * piv.shape[1] + 2, 0.7 * piv.shape[0] + 1.8))
im = ax.imshow(piv.values, cmap="viridis", vmin=max(0.5, np.nanmin(piv.values) - 0.02), vmax=1)
ax.set_xticks(range(piv.shape[1]), piv.columns, rotation=30, ha="right")
ax.set_yticks(range(piv.shape[0]), piv.index)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        ax.text(j, i, f"{piv.values[i, j]:.3f}", ha="center", va="center", color="w", fontsize=9)
ax.set_title("AUC out-of-fold — embeddings congelados + LogReg")
plt.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout()
plt.savefig(OUT / "fig" / f"auc_backbone_mapa{TAG}.png", dpi=110)
plt.show()

# elección del backbone SOLO con OOF (el test no participa)
clave = lambda bb: (round(RESULTADOS[f"EMB_LR_{CORTO[bb]}_fusprom"]["oof"]["auc"], 3),
                    RESULTADOS[f"EMB_LR_{CORTO[bb]}_fusprom"]["oof"]["esp"])
BEST_BB = max(BACKBONES, key=clave)
BEST_SIZE = BACKBONES[BEST_BB]
print("backbone elegido (AUC OOF de la fusión, desempate especificidad OOF):", BEST_BB, clave(BEST_BB))

## Fase 6 — Transfer learning del mejor backbone (solo cabeza), por mapa, con/sin augmentación

Backbone congelado y cabeza `Dropout(0.3) → Linear(1)`, con `BCEWithLogits(pos_weight)`, Adam y
early stopping por **AUC de validación** del fold. Se entrena un modelo por fold: la predicción de
validación arma el OOF, y el test recibe el **promedio de los modelos de los folds**.

In [ ]:
def ruta_ckpt(exp_id, k):
    return OUT / "ckpt" / f"{exp_id}_f{k}{TAG}.pth"


def descongelar(model, n_blocks):
    for p in model.parameters():
        p.requires_grad = False
    bb = model.backbone
    if hasattr(bb, "blocks"):                         # MobileNetV2 / EfficientNet
        partes = list(bb.blocks)[-n_blocks:]
    else:                                             # ConvNeXt: 4 stages -> se libera el último
        partes = list(bb.stages)[-1:]
    extras = [getattr(bb, nm) for nm in ("conv_head", "bn2", "head", "norm_pre")
              if isinstance(getattr(bb, nm, None), nn.Module)]
    for m in partes + extras + [model.head]:
        for p in m.parameters():
            p.requires_grad = True


def entrenar_cv(exp_id, bb, size, mapa, aug, modo="cabeza", init_id=None):
    r = cargar_res(exp_id)
    if r and all(ruta_ckpt(exp_id, k).exists() for k in FOLDS):
        return r
    cfg = HEAD_CFG if modo == "cabeza" else FT_CFG
    imgs = imagenes_mapa(mapa, size)
    oof, p_test, hist_all, t0 = np.full(len(IDX_POOL), np.nan), [], {}, time.time()
    for k in FOLDS:
        set_seed(SEED + k)
        model = Clf(bb).to(DEVICE)
        if modo == "cabeza":
            for p in model.backbone.parameters():
                p.requires_grad = False
            opt = torch.optim.Adam(model.head.parameters(), lr=cfg["lr"])
            sched = None
        else:
            model.load_state_dict(torch.load(ruta_ckpt(init_id, k), map_location=DEVICE))
            descongelar(model, cfg["unfreeze_blocks"])
            bb_params = [p for p in model.backbone.parameters() if p.requires_grad]
            opt = torch.optim.AdamW([dict(params=bb_params, lr=cfg["lr_backbone"]),
                                     dict(params=list(model.head.parameters()), lr=cfg["lr_head"])],
                                    weight_decay=cfg["weight_decay"])
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])
        tr_idx, va_mask = IDX_POOL[FOLD_POOL != k], FOLD_POOL == k
        va_idx = IDX_POOL[va_mask]
        n1 = int(Y[tr_idx].sum())
        crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor((len(tr_idx) - n1) / max(n1, 1), device=DEVICE))
        tf_ev = build_tf(size, False, model.mean, model.std)
        dl_tr = DataLoader(MapaDS(imgs, tr_idx, build_tf(size, aug, model.mean, model.std)),
                           batch_size=cfg["batch"], shuffle=True, num_workers=NUM_WORKERS,
                           generator=torch.Generator().manual_seed(SEED + k))
        ds_va = MapaDS(imgs, va_idx, tf_ev)
        scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
        best_auc, best_state, wait, hist = -1.0, None, 0, []
        for ep in range(1, cfg["epochs"] + 1):
            model.train()
            ep_loss, n_vistos, t_ep = 0.0, 0, time.time()
            for x, yb in dl_tr:
                x, yb = x.to(DEVICE), yb.to(DEVICE)
                opt.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                    loss = crit(model(x).float(), yb)
                if not torch.isfinite(loss):
                    continue
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
                ep_loss += loss.item() * len(yb)
                n_vistos += len(yb)
            if sched is not None:
                sched.step()
            auc = float(roc_auc_score(Y[va_idx], predecir(model, ds_va)))
            hist.append(dict(epoch=ep, train_loss=round(ep_loss / max(n_vistos, 1), 4),
                             val_auc=round(auc, 4), s=round(time.time() - t_ep, 1)))
            if best_state is None or auc > best_auc + 1e-4:
                best_auc, wait = auc, 0
                best_state = {kk: v.detach().cpu().clone() for kk, v in model.state_dict().items()}
            else:
                wait += 1
                if wait >= cfg["patience"]:
                    break
        torch.save(best_state, ruta_ckpt(exp_id, k))
        model.load_state_dict(best_state)
        oof[va_mask] = predecir(model, ds_va)
        p_test.append(predecir(model, MapaDS(imgs, IDX_TEST, tf_ev)))
        hist_all[k] = hist
        print(f"  {exp_id} fold {k}: {len(hist)} ép., mejor val AUC = {best_auc:.4f}", flush=True)
        del model, opt
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    return armar_res(exp_id, "transfer learning (cabeza)" if modo == "cabeza" else "fine-tuning",
                     oof, np.mean(p_test, 0),
                     dict(backbone=bb, mapa=mapa, aug=aug, modo=modo, init=init_id,
                          tiempo_s=round(time.time() - t0, 1), historia=hist_all))

In [ ]:
c = CORTO[BEST_BB]
IDS_CAB, MEJOR_CAB = [], {}
for mapa in MAPAS:
    for aug in AUG_BRAZOS:
        eid = f"CAB_{c}_{mapa}_{'aug' if aug else 'noaug'}"
        RESULTADOS[eid] = entrenar_cv(eid, BEST_BB, BEST_SIZE, mapa, aug, "cabeza")
        IDS_CAB.append(eid)
    # brazo elegido por AUC OOF (no por test)
    MEJOR_CAB[mapa] = max([i for i in IDS_CAB if f"_{mapa}_" in i], key=lambda i: RESULTADOS[i]["oof"]["auc"])
RESULTADOS[f"CAB_{c}_fusprom"] = fusionar(f"CAB_{c}_fusprom", list(MEJOR_CAB.values()),
                                          "cabeza, promedio 4 mapas", dict(backbone=BEST_BB, mapa="prom4"))
IDS_CAB.append(f"CAB_{c}_fusprom")
print("brazo elegido por mapa:", MEJOR_CAB)
display(tabla(IDS_CAB)[["ID", "mapa", "AUC_cv", "AUC_cv_sd", "AUC_oof", "Esp_oof", "AUC_test",
                        "Sens_test", "Esp_test", "Esp_a_sens95_test"]].round(4))

In [ ]:
fig, axes = plt.subplots(1, len(MAPAS), figsize=(4.2 * len(MAPAS), 3.4), squeeze=False)
for ax, mapa in zip(axes[0], MAPAS):
    for aug in AUG_BRAZOS:
        h = RESULTADOS[f"CAB_{c}_{mapa}_{'aug' if aug else 'noaug'}"]["extra"]["historia"]
        for k, hk in h.items():
            ax.plot([e["epoch"] for e in hk], [e["val_auc"] for e in hk],
                    color="tab:orange" if aug else "tab:blue", alpha=0.6,
                    label=("con aug" if aug else "sin aug") if str(k) == str(FOLDS[0]) else None)
    ax.set_title(f"{mapa}: AUC val por fold")
    ax.set_xlabel("época")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Fase 7 — Fine-tuning del mejor backbone: evaluación antes / después

Se parte del checkpoint de la cabeza (brazo elegido por mapa y por fold) y se descongelan los
**últimos `unfreeze_blocks` bloques** (+ `conv_head`/`bn2`; en ConvNeXt, el último stage). Se usa
augmentación, AdamW con weight decay, LR diferencial (backbone 1e-4 / cabeza 3e-4), cosine y early
stopping por AUC de validación. "Antes" y "después" se evalúan con la **misma** función, los mismos
folds y el mismo test.

In [ ]:
IDS_FT = []
for mapa in MAPAS_FT:
    eid = f"FT_{c}_{mapa}"
    RESULTADOS[eid] = entrenar_cv(eid, BEST_BB, BEST_SIZE, mapa, FT_CFG["aug"], "ft", init_id=MEJOR_CAB[mapa])
    IDS_FT.append(eid)
if len(MAPAS_FT) > 1:
    RESULTADOS[f"FT_{c}_fusprom"] = fusionar(f"FT_{c}_fusprom", IDS_FT, "fine-tuning, promedio de mapas",
                                             dict(backbone=BEST_BB, mapa=f"prom{len(MAPAS_FT)}"))
    IDS_FT.append(f"FT_{c}_fusprom")

filas_ab = []
for mapa in MAPAS_FT:
    for etapa, i in [("antes (cabeza)", MEJOR_CAB[mapa]), ("después (FT)", f"FT_{c}_{mapa}")]:
        f_ = fila(RESULTADOS[i])
        filas_ab.append(dict(mapa=mapa, etapa=etapa, ID=i, AUC_oof=f_["AUC_oof"], Esp_oof=f_["Esp_oof"],
                             AUC_test=f_["AUC_test"], Sens_test=f_["Sens_test"], Esp_test=f_["Esp_test"],
                             Esp_a_sens95_test=f_["Esp_a_sens95_test"], Brier_test=f_["Brier_test"]))
display(pd.DataFrame(filas_ab).round(4))

## Fase 8 — Tabla final, elección del modelo y curvas ROC

* Para **cada** experimento, `tau` se calibró con OOF (sensibilidad OOF ≥ 0.95) y se aplicó **sin tocar** al test.
* Se agrega una fusión "**cabeza + FT**" (las 8 probabilidades por mapa promediadas).
* El **modelo elegido** es el de mayor especificidad OOF @tau, con desempate por AUC OOF. **Solo train/OOF**: el test se mira únicamente para reportar.

In [ ]:
if len(MAPAS_FT) > 1:
    eid = f"CABFT_{c}_fusprom"
    RESULTADOS[eid] = fusionar(eid, list(MEJOR_CAB.values()) + [f"FT_{c}_{m}" for m in MAPAS_FT],
                               "cabeza + FT, promedio", dict(backbone=BEST_BB, mapa="prom"))
    IDS_FT.append(eid)

ORDEN = ["TAB_LR"] + IDS_EMB + IDS_CAB + IDS_FT
df_res = tabla(ORDEN)
cand = df_res[df_res.ID != "TAB_LR"]
MEJOR = cand.sort_values(["Esp_oof", "AUC_oof"], ascending=False).iloc[0]["ID"]
print("MODELO ELEGIDO (por OOF):", MEJOR)
cols = ["ID", "familia", "mapa", "AUC_oof", "tau", "Esp_oof", "AUC_test", "AUC_ic", "Sens_test",
        "Sens_ic_wilson", "Esp_test", "Esp_ic_boot", "Esp_a_sens95_test", "AP_test", "Brier_test", "FN_test"]
display(df_res[cols].round(4))

In [ ]:
top = df_res.sort_values("AUC_oof", ascending=False).ID.head(5).tolist()
top = list(dict.fromkeys(top + ["TAB_LR", MEJOR]))
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
ax = axes[0]
for i in top:
    r = RESULTADOS[i]
    fpr, tpr, _ = roc_curve(Y_TEST, r["test_p"])
    ax.plot(fpr, tpr, lw=2 if i == MEJOR else 1.2, label=f"{i} (AUC={r['test']['auc']:.3f})")
    ax.scatter([1 - r["test"]["esp"]], [r["test"]["sens"]], s=35, zorder=5)
ax.axhline(OBJ_SENS, ls="--", color="gray")
ax.plot([0, 1], [0, 1], ":", color="lightgray")
ax.set_xlabel("1 - especificidad")
ax.set_ylabel("sensibilidad (recall QC)")
ax.set_title("ROC en test — puntos = operación con tau de OOF")
ax.legend(fontsize=7, loc="lower right")

ax = axes[1]
d = df_res.set_index("ID").loc[top]
x = np.arange(len(d))
sens_err = np.array([[RESULTADOS[i]["test"]["sens"] - RESULTADOS[i]["test"]["sens_ic"][0] for i in top],
                     [RESULTADOS[i]["test"]["sens_ic"][1] - RESULTADOS[i]["test"]["sens"] for i in top]])
sens_err = np.clip(sens_err, 0, None)        # redondeo: con sens = 1 la cota de Wilson da 0.9999...
ax.bar(x - 0.2, d["Sens_test"], 0.4, yerr=sens_err, capsize=3, color="tab:red", label="sensibilidad @tau (IC Wilson)")
ax.bar(x + 0.2, d["Esp_test"], 0.4, color="tab:blue", label="especificidad @tau")
ax.axhline(OBJ_SENS, ls="--", color="gray")
ax.set_xticks(x, top, rotation=40, ha="right", fontsize=8)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=8, loc="lower right")
ax.set_title("Punto de operación en test")
plt.tight_layout()
plt.savefig(OUT / "fig" / f"roc_operacion{TAG}.png", dpi=110)
plt.show()

## Fase 9 — Explicabilidad: Grad-CAM antes y después del fine-tuning

Se usa Grad-CAM con hooks manuales sobre el último mapa de activación del backbone
(`forward_features`). Para cada mapa se toman imágenes del test (QC y sanas) y se explican con el
modelo de **cabeza** y el de **FT** del primer fold. Hay que revisar especialmente el **Axial**: si el
calor cae sobre los **números impresos** y no sobre la zona de encurvamiento, el modelo está "leyendo"
Kmax (atajo). La interpretación se redacta a mano a partir de las figuras.

In [ ]:
def gradcam(model, x):
    model.eval()
    x = x.clone().requires_grad_(True)
    feats = model.backbone.forward_features(x)
    feats.retain_grad()
    logit = model.head(model.backbone.forward_head(feats, pre_logits=True)).squeeze()
    model.zero_grad(set_to_none=True)
    logit.backward()
    w = feats.grad.mean(dim=(2, 3), keepdim=True)
    cam = torch.relu((w * feats).sum(1))[0].detach().cpu().numpy()
    return cam / (cam.max() + 1e-8), float(torch.sigmoid(logit).item())


def cargar_modelo(exp_id, k):
    m = Clf(BEST_BB).to(DEVICE)
    m.load_state_dict(torch.load(ruta_ckpt(exp_id, k), map_location=DEVICE))
    return m.eval()


rng = np.random.default_rng(SEED)
sel_cam = np.concatenate([rng.choice(IDX_TEST[Y_TEST == lab], GRADCAM_POR_CLASE, replace=False) for lab in (1, 0)])
k0 = FOLDS[0]
for mapa in MAPAS_FT:
    modelos = [("cabeza", cargar_modelo(MEJOR_CAB[mapa], k0)), ("FT", cargar_modelo(f"FT_{c}_{mapa}", k0))]
    imgs = imagenes_mapa(mapa, BEST_SIZE)
    tf_ev = build_tf(BEST_SIZE, False, modelos[0][1].mean, modelos[0][1].std)
    fig, axes = plt.subplots(len(sel_cam), 3, figsize=(10, 3.4 * len(sel_cam)), squeeze=False)
    for fi, j in enumerate(sel_cam):
        vis = np.asarray(T.CenterCrop(BEST_SIZE)(imgs[j]))
        axes[fi, 0].imshow(vis)
        axes[fi, 0].set_title(f"{man.eye_key[j]} | real={'QC' if Y[j] else 'sano'}", fontsize=9)
        for ci, (nom, mdl) in enumerate(modelos, start=1):
            cam, prob = gradcam(mdl, tf_ev(imgs[j]).unsqueeze(0).to(DEVICE))
            cam_img = np.asarray(Image.fromarray(np.uint8(255 * cam)).resize((BEST_SIZE, BEST_SIZE), Image.BILINEAR))
            axes[fi, ci].imshow(vis)
            axes[fi, ci].imshow(cam_img, cmap="jet", alpha=0.45)
            axes[fi, ci].set_title(f"{nom}: P(QC)={prob:.2f}", fontsize=9)
        for ax in axes[fi]:
            ax.axis("off")
    fig.suptitle(f"Grad-CAM — {mapa} ({CORTO[BEST_BB]}, fold {k0})", fontsize=12)
    plt.tight_layout()
    plt.savefig(OUT / "fig" / f"gradcam_{mapa}{TAG}.png", dpi=90)
    plt.show()
    del modelos

## Fase 10 — Desempeño por subgrupo (género y ojo) del modelo elegido

Lo pide el EDA §7: el género está desbalanceado (65 % mujeres). Se reportan sensibilidad y
especificidad @tau con IC de Wilson. Con n chico por subgrupo, las diferencias menores que el IC
**no** son concluyentes.

In [ ]:
r = RESULTADOS[MEJOR]
p_te, tau = np.array(r["test_p"]), r["test"]["tau"]
filas_sg = []
for var in ("gender", "eye"):
    for val in sorted(man[var].iloc[IDX_TEST].unique()):
        m = man[var].to_numpy(dtype=str)[IDX_TEST] == val
        mt = metricas_bin(Y_TEST[m], p_te[m], tau)
        filas_sg.append(dict(variable=var, valor=val, n=int(m.sum()), n_QC=int(Y_TEST[m].sum()),
                             sens=mt["sens"], sens_ic=f"[{mt['sens_ic'][0]:.3f}-{mt['sens_ic'][1]:.3f}]",
                             esp=mt["esp"], esp_ic=f"[{mt['esp_ic'][0]:.3f}-{mt['esp_ic'][1]:.3f}]",
                             auc=mt["auc"]))
display(pd.DataFrame(filas_sg).round(4))

## Fase 11 — Resumen y exportación

In [ ]:
b, t, o = RESULTADOS[MEJOR]["test_boot"], RESULTADOS[MEJOR]["test"], RESULTADOS[MEJOR]["oof"]
tb = RESULTADOS["TAB_LR"]["test"]
lineas = [
    f"### Resultados — análisis de backbones (queratocono){' — SMOKE' if SMOKE else ''}",
    "",
    f"* Ojos: train {len(IDX_POOL)} (CV {len(FOLDS)} folds oficiales) / test {len(IDX_TEST)}. "
    f"Objetivo: sensibilidad OOF ≥ {OBJ_SENS}.",
    f"* **Backbone elegido** (AUC OOF de la fusión de embeddings): `{BEST_BB}`.",
    f"* **Modelo elegido por OOF**: `{MEJOR}` → tau = {o['tau']:.3f}; en test: "
    f"sensibilidad = {t['sens']:.3f} [Wilson {t['sens_ic'][0]:.3f}-{t['sens_ic'][1]:.3f}], "
    f"especificidad = {t['esp']:.3f} [boot {b['esp'][0]:.3f}-{b['esp'][1]:.3f}], "
    f"AUC = {t['auc']:.3f} [boot {b['auc'][0]:.3f}-{b['auc'][1]:.3f}], FN = {t['fn']}, FP = {t['fp']}.",
    f"* **Baseline tabular** (LogReg clínica): AUC test = {tb['auc']:.3f}, sensibilidad = {tb['sens']:.3f}, "
    f"especificidad = {tb['esp']:.3f}.",
    "* Grad-CAM: figuras en `fig/gradcam_*.png`. Su interpretación (mapa frente a números impresos) se redacta a mano.",
    "* Limitaciones: el test tiene ~110 QC, así que el IC de la sensibilidad es de ±3-5 puntos y las diferencias menores no son concluyentes.",
]
txt = "\n".join(lineas)
display(Markdown(txt))

csv_out = OUT / f"02_resultados_backbone{TAG}.csv"
df_res.to_csv(csv_out, index=False)
(OUT / f"02_resumen{TAG}.md").write_text(txt + "\n", encoding="utf-8")
print("CSV:", csv_out.resolve())
print("resumen:", (OUT / f"02_resumen{TAG}.md").resolve())

## Reproducibilidad

* **Semilla 42** en el muestreo, los DataLoaders (`generator`), cada fold (`SEED + k`), LogReg/SVM y Grad-CAM.
* **Splits oficiales** del dataset (`split_80_20`, `fold`). Se verifica por código que no hay paciente en dos folds ni en train y test a la vez.
* **Qué se calibra con qué**: la selección de época, brazo, backbone y modelo, y el umbral `tau`, usan **solo OOF/validación**. El test (promedio de los modelos de cada fold) solo se usa para reportar.
* **Caché**: `res/*.json` (con probabilidades y claves de ojo para verificar la alineación), `ckpt/*.pth` y `emb/*.npz`. Si existen, no se recalculan. En Colab viven en Drive y sobreviven al corte de sesión.
* **Smoke test**: `SMOKE=1 jupyter nbconvert --to notebook --execute 02_Analisis_BackBone.ipynb --output 02_smoke_out.ipynb`.